# XGBoost + PCA + Optuna — LRG

Versao otimizada do [../04_xgboost_baseline/xgb_pca_didatico.ipynb](../04_xgboost_baseline/xgb_pca_didatico.ipynb).

**Pipeline**: espectro normalizado -> PCA(300) + scalars(4) -> StandardScaler -> XGBoost tunado com Optuna (TPE + SQLite, pode pausar e continuar).

**Alvo**: NMAD ~0.020-0.022 (entre o didatico 0.0286 e o xgb_lrg cru 0.0180), mantendo interpretabilidade via feature importance.

**Storage SQLite**: o estudo persiste em `models/LRG/xgb_optuna_pca/optuna_pca_lrg.db`. Pode interromper (Ctrl-C) e re-rodar a celula 7 — vai retomar do trial onde parou.

## 1. Parametros (papermill)

In [ ]:
OBJECT_TYPE   = 'LRG'
SEED          = 42
N_COMPONENTS  = 300       # mesmo valor que ficou bom no didatico
TEST_SIZE     = 0.15
VAL_SIZE      = 0.15
N_TRIALS      = 50        # Optuna; suba pra 100+ se tiver tempo

## 2. Imports e setup

In [ ]:
import sys, os, time, json, random
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
import xgboost as xgb
import optuna
import joblib

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / 'config.py').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import paths_for, RESULTS_DIR, MODELS_DIR, SPLITS_DIR
from src.data import load_spectral_dataset, normalize_spectra, make_or_load_split, make_strat_bins
from src.models import tune_xgboost_with_optuna
from src.evaluation.metrics import (
    compute_redshift_metrics, delta_z_normalized, sigma_nmad,
    metrics_by_redshift_bin,
)

random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid')

print(f'numpy   : {np.__version__}')
print(f'xgboost : {xgb.__version__}')
print(f'optuna  : {optuna.__version__}')
print(f'objeto  : {OBJECT_TYPE}  | N_TRIALS={N_TRIALS}  | N_COMPONENTS={N_COMPONENTS}')

## 3. Carrega espectros, extrai scalars, normaliza

Mesma logica do `xgb_pca_didatico`: pega 4 scalars do bruto antes de normalizar (essa info se perde apos a normalizacao).

In [ ]:
def extract_scalars(X_orig):
    X_abs = np.abs(X_orig)
    X_pos = np.where(X_orig > 0, X_orig, np.nan)
    with np.errstate(divide='ignore', invalid='ignore'):
        log_max    = np.log10(X_abs.max(axis=1))
        log_median = np.log10(np.nanmedian(X_pos, axis=1))
        log_sum    = np.log10(X_abs.sum(axis=1))
        log_p95    = np.log10(np.nanpercentile(X_pos, 95, axis=1))
    scalars = np.stack([log_max, log_median, log_sum, log_p95], axis=1)
    scalars = np.where(np.isfinite(scalars), scalars, -40.0)
    return scalars.astype(np.float32)

paths = paths_for(OBJECT_TYPE)
HDF5 = paths['spectra_h5'].with_name(f'{OBJECT_TYPE}spectra_padded.h5')
print(f'HDF5: {HDF5}')

X_orig, y_all, n_wave = load_spectral_dataset(HDF5)
scalars_all = extract_scalars(X_orig)
X_spec = normalize_spectra(X_orig).astype(np.float32)
y_all = y_all.astype(np.float32)
del X_orig

print(f'X_spec      : {X_spec.shape}')
print(f'scalars_all : {scalars_all.shape}')
print(f'y_all       : {y_all.shape}  z em [{y_all.min():.3f}, {y_all.max():.3f}]')

## 4. Splits 3-way estratificados + PCA + StandardScaler

In [ ]:
# Split canonico ESTRATIFICADO por z (splits/<OBJ>_split.npz). Fonte unica:
# src/data/splits.py — mesma politica que era feita aqui inline.
train_idx, val_idx, test_idx = make_or_load_split(OBJECT_TYPE, y_all, SPLITS_DIR)

# pool/relativos + q (mantidos pra split_idx.npz e run_info downstream)
pool_idx      = np.concatenate([train_idx, val_idx])
train_in_pool = np.arange(len(train_idx))
val_in_pool   = np.arange(len(train_idx), len(pool_idx))
y_pool = y_all[pool_idx]
_, q_outer = make_strat_bins(y_all)
_, q_inner = make_strat_bins(y_pool)

X_pool = X_spec[pool_idx]; S_pool = scalars_all[pool_idx]
X_train, X_val, X_test = X_spec[train_idx], X_spec[val_idx], X_spec[test_idx]
y_train, y_val, y_test = y_all[train_idx],  y_all[val_idx],  y_all[test_idx]
S_train, S_val, S_test = scalars_all[train_idx], scalars_all[val_idx], scalars_all[test_idx]

print(f'Train : n = {len(y_train):>7,}  z_mean = {y_train.mean():.4f}')
print(f'Val   : n = {len(y_val):>7,}  z_mean = {y_val.mean():.4f}')
print(f'Test  : n = {len(y_test):>7,}  z_mean = {y_test.mean():.4f}')

# PCA — fit so no train
print(f'\nPCA com {N_COMPONENTS} componentes...')
t0 = time.time()
pca = PCA(n_components=N_COMPONENTS, random_state=SEED)
P_train = pca.fit_transform(X_train)
P_val   = pca.transform(X_val)
P_test  = pca.transform(X_test)
print(f'  feito em {time.time()-t0:.1f}s | variancia retida: {pca.explained_variance_ratio_.sum()*100:.2f}%')

# Concat + StandardScaler (fit so no train)
X_train_full = np.concatenate([P_train, S_train], axis=1)
X_val_full   = np.concatenate([P_val,   S_val  ], axis=1)
X_test_full  = np.concatenate([P_test,  S_test ], axis=1)

scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train_full)
X_val_std   = scaler.transform(X_val_full)
X_test_std  = scaler.transform(X_test_full)

feature_names = [f'PC{i+1}' for i in range(N_COMPONENTS)] + ['log_max', 'log_median', 'log_sum', 'log_p95']
print(f'\nX_train_std : {X_train_std.shape}  ({N_COMPONENTS} PCs + 4 scalars)')

## 5. Tuning com Optuna (SQLite, retomavel)

Usa `tune_xgboost_with_optuna` do `src.models`. A funcao:
- otimiza `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`, `min_child_weight`, `reg_alpha`, `reg_lambda`, `gamma`
- usa **TPE** (sampler bayesiano)
- persiste no SQLite com `load_if_exists=True` -> pode pausar e continuar
- objetivo: minimizar **RMSE no val**

Tempo aproximado: ~30-60s por trial com 300 features e early stopping. 50 trials = ~25-50 min.

In [ ]:
RUN_TAG = f'xgb_optuna_pca-{OBJECT_TYPE}-pc{N_COMPONENTS}_t{N_TRIALS}_s{SEED}'
RUN_DIR = RESULTS_DIR / OBJECT_TYPE / 'xgb_optuna_pca' / RUN_TAG
MODEL_DIR = MODELS_DIR / OBJECT_TYPE / 'xgb_optuna_pca' / RUN_TAG
for d in (RUN_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

storage = f'sqlite:///{MODEL_DIR / f"optuna_pca_{OBJECT_TYPE.lower()}.db"}'
study_name = f'xgb_pca_{OBJECT_TYPE.lower()}'

print(f'Storage: {storage}')
print(f'Study  : {study_name}')
print(f'Trials : {N_TRIALS}  (se ja existir, retoma de onde parou)\n')

t0 = time.time()
study = tune_xgboost_with_optuna(
    X_train_std, y_train, X_val_std, y_val,
    n_trials=N_TRIALS,
    study_name=study_name,
    storage=storage,
    seed=SEED,
    show_progress=True,
)
tuning_time = time.time() - t0

print(f'\nTempo total de tuning: {tuning_time/60:.1f} min')
print(f'Melhor RMSE no val   : {study.best_value:.5f}')
print(f'Melhor trial         : #{study.best_trial.number}')
print(f'\nMelhores HPs:')
for k, v in study.best_params.items():
    if isinstance(v, float):
        print(f'  {k:20} = {v:.6g}')
    else:
        print(f'  {k:20} = {v}')

## 6. Visualizacoes do Optuna

- **history**: como o melhor RMSE evoluiu com os trials. Se ainda esta caindo perto do fim, vale rodar mais trials.
- **importance**: quais HPs mais impactam o RMSE. Foca futuras buscas nesses.

In [ ]:
from optuna.importance import get_param_importances

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

# History
trials_df = study.trials_dataframe()
running_best = trials_df['value'].cummin()
axes[0].scatter(trials_df['number'], trials_df['value'], alpha=0.5, s=30, label='trial')
axes[0].plot(trials_df['number'], running_best, color='red', lw=2, label='melhor ate agora')
axes[0].axhline(study.best_value, color='red', ls=':', alpha=0.5)
axes[0].set_xlabel('# do trial'); axes[0].set_ylabel('RMSE no val')
axes[0].set_title(f'Historico Optuna  (best = {study.best_value:.5f})')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Param importance
try:
    imp = get_param_importances(study)
    names = list(imp.keys())[::-1]; vals = list(imp.values())[::-1]
    axes[1].barh(names, vals, color='steelblue')
    axes[1].set_xlabel('importance')
    axes[1].set_title('Importance dos HPs no RMSE val')
    axes[1].grid(axis='x', alpha=0.3)
except Exception as e:
    axes[1].text(0.5, 0.5, f'param importance indisponivel\n({e})',
                 transform=axes[1].transAxes, ha='center', va='center')

plt.tight_layout()
plt.show()

## 7. Treino final com os melhores HPs

Treina com **train + val combinados** e os melhores HPs. `n_estimators` final = melhor do study (Optuna ja escolheu via early stopping interno).

In [ ]:
best_params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': 'cpu',
    'random_state': SEED,
    'n_jobs': -1,
    **study.best_params,
}

# Refit em train+val
X_trainval = np.concatenate([X_train_std, X_val_std], axis=0)
y_trainval = np.concatenate([y_train, y_val], axis=0)

t0 = time.time()
final = xgb.XGBRegressor(**best_params)
final.fit(X_trainval, y_trainval, verbose=False)
final_train_time = time.time() - t0
print(f'Treino final: {final_train_time:.1f}s | n = {len(y_trainval):,}')

## 8. Avaliacao no test

In [ ]:
y_pred_test = final.predict(X_test_std)
dz_test = delta_z_normalized(y_test, y_pred_test)

test_metrics = {
    'mae':               float(np.mean(np.abs(y_pred_test - y_test))),
    'rmse':              float(np.sqrt(np.mean((y_pred_test - y_test)**2))),
    'bias':              float(np.median(dz_test)),
    'sigma_nmad':        sigma_nmad(dz_test),
    'eta_outliers_0.05': float(100.0 * np.mean(np.abs(dz_test) > 0.05)),
    'eta_outliers_0.15': float(100.0 * np.mean(np.abs(dz_test) > 0.15)),
}

print('=' * 60)
print(f'RESULTADO NO TEST  (n = {len(y_test):,}, objeto = {OBJECT_TYPE})')
print('=' * 60)
for k, v in test_metrics.items():
    print(f'  {k:>22} = {v:.4f}')

# Comparacao rapida com referencias
print('\nComparacao com baselines:')
print(f'  xgb_lrg cru (4674 feat):     NMAD = 0.0180')
print(f'  xgb_pca_didatico (300 PCs):  NMAD = 0.0286')
print(f'  ESTE (PCA + Optuna):         NMAD = {test_metrics["sigma_nmad"]:.4f}')

## 9. Plots para o TCC

Os mesmos 4 paineis do `xgb_pca_didatico` secao 11 — pra empilhar com CNN no notebook de comparacao.

In [ ]:
OUT_THR = 0.15
is_outlier = (np.abs(dz_test) > OUT_THR).astype(float)

bin_dict = metrics_by_redshift_bin(y_test, y_pred_test, n_bins=12, outlier_threshold=OUT_THR)
bin_df = pd.DataFrame(bin_dict['bins'])

abs_dz_sorted = np.sort(np.abs(dz_test))
cdf = np.arange(1, len(abs_dz_sorted) + 1) / len(abs_dz_sorted)

z_lo, z_hi = float(y_test.min()), float(y_test.max())
extent = [z_lo, z_hi, z_lo, z_hi]

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# (a) hexbin log
ax = axes[0, 0]
hb = ax.hexbin(y_test, y_pred_test, gridsize=50, cmap='viridis',
               mincnt=1, norm=LogNorm(), extent=extent)
ax.plot([z_lo, z_hi], [z_lo, z_hi], 'r--', lw=1.5, label='ideal')
zz = np.linspace(z_lo, z_hi, 100)
ax.plot(zz, zz + OUT_THR*(1+zz), 'w:', lw=1, alpha=0.7, label=f'+/-{OUT_THR}*(1+z)')
ax.plot(zz, zz - OUT_THR*(1+zz), 'w:', lw=1, alpha=0.7)
ax.set_xlabel('z_real'); ax.set_ylabel('z_predito')
ax.set_title(f'(a) z_pred vs z_real (log)  sigma_NMAD={test_metrics["sigma_nmad"]:.4f}')
ax.set_xlim(z_lo, z_hi); ax.set_ylim(z_lo, z_hi)
ax.legend(loc='lower right', fontsize=9)
plt.colorbar(hb, ax=ax, label='# galaxias (log)')

# (b) outliers por regiao
ax = axes[0, 1]
hb = ax.hexbin(y_test, y_pred_test, C=is_outlier*100.0,
               reduce_C_function=np.mean, gridsize=50, cmap='magma',
               mincnt=5, vmin=0, vmax=100, extent=extent)
ax.plot([z_lo, z_hi], [z_lo, z_hi], 'c--', lw=1.5, label='ideal')
ax.set_xlabel('z_real'); ax.set_ylabel('z_predito')
ax.set_title(f'(b) Outliers por regiao  total={test_metrics["eta_outliers_0.15"]:.2f}%')
ax.set_xlim(z_lo, z_hi); ax.set_ylim(z_lo, z_hi)
ax.legend(loc='lower right', fontsize=9)
plt.colorbar(hb, ax=ax, label='% outliers na celula')

# (c) NMAD/bias por bin
ax = axes[1, 0]; ax2 = ax.twinx()
l1 = ax.plot(bin_df['z_center'], bin_df['nmad'], 'o-', color='steelblue', lw=2, label='sigma_NMAD')
l2 = ax2.plot(bin_df['z_center'], bin_df['bias'], 's--', color='darkorange', lw=1.5, label='bias')
ax2.axhline(0, color='gray', lw=0.5)
ax3 = ax.twinx()
ax3.bar(bin_df['z_center'], bin_df['n'],
        width=(bin_df['z_high']-bin_df['z_low'])*0.9, alpha=0.12, color='gray', zorder=0)
ax3.set_ylabel('# galaxias no bin', color='gray', fontsize=9)
ax3.tick_params(axis='y', labelcolor='gray', labelsize=8)
ax3.spines['right'].set_position(('outward', 50))
ax.set_xlabel('z_real (centro do bin)')
ax.set_ylabel('sigma_NMAD', color='steelblue')
ax2.set_ylabel('bias', color='darkorange')
ax.tick_params(axis='y', labelcolor='steelblue')
ax2.tick_params(axis='y', labelcolor='darkorange')
ax.set_title('(c) sigma_NMAD e bias por bin de z')
ax.legend(l1+l2, [l.get_label() for l in l1+l2], loc='upper left', fontsize=9)
ax.grid(alpha=0.3)

# (d) CDF
ax = axes[1, 1]
ax.plot(abs_dz_sorted, cdf*100, lw=2, color='steelblue', label='XGB PCA Optuna')
for thr, col in [(0.05, 'orange'), (OUT_THR, 'red')]:
    frac = float(np.mean(np.abs(dz_test) <= thr) * 100)
    ax.axvline(thr, color=col, ls='--', lw=1.2)
    ax.text(thr*1.05, 5, f'|dz|={thr}\n{frac:.1f}% abaixo', color=col, fontsize=9)
ax.set_xlim(0, 0.3)
ax.set_xlabel('|delta_z / (1+z)|'); ax.set_ylabel('% cumulativo de galaxias')
ax.set_title('(d) CDF do |delta_z_norm|')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 10. Salvar tudo

In [ ]:
# Workaround pra bug do XGBoost 3.2.0 + sklearn novo: _estimator_type
# nao esta sendo herdado do RegressorMixin. save_model() falha sem isso.
final._estimator_type = 'regressor'

# Modelo + PCA + scaler
final.save_model(MODEL_DIR / 'xgb_model.json')
joblib.dump(pca,     MODEL_DIR / 'pca.pkl')
joblib.dump(scaler,  MODEL_DIR / 'scaler.pkl')

# Predicoes + splits + bins (pra overlay com CNN)
np.savez(RUN_DIR / 'test_outputs.npz',
         y_test=y_test, y_pred=y_pred_test, delta_z=dz_test)
np.savez(RUN_DIR / 'split_idx.npz',
         pool_idx=pool_idx, test_idx=test_idx,
         train_in_pool=train_in_pool, val_in_pool=val_in_pool)
bin_df.to_csv(RUN_DIR / 'metrics_by_zbin.csv', index=False)

# Feature importance
importances = pd.Series(final.feature_importances_, index=feature_names).sort_values(ascending=False)
importances.to_csv(RUN_DIR / 'feature_importance.csv', header=['importance_gain'])

# Optuna study (CSV pra inspecao manual)
study.trials_dataframe().to_csv(RUN_DIR / 'optuna_trials.csv', index=False)

run_info = {
    'created_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
    'run_tag': RUN_TAG,
    'object_type': OBJECT_TYPE,
    'seed': SEED,
    'n_components_pca': N_COMPONENTS,
    'pca_variance_retained': float(pca.explained_variance_ratio_.sum()),
    'n_trials': N_TRIALS,
    'best_trial_number': int(study.best_trial.number),
    'best_val_rmse': float(study.best_value),
    'best_params': study.best_params,
    'tuning_time_min': float(tuning_time / 60),
    'final_train_time_s': float(final_train_time),
    'splits': {'n_train': int(len(y_train)), 'n_val': int(len(y_val)), 'n_test': int(len(y_test))},
    'test_metrics': test_metrics,
    'storage': storage,
    'study_name': study_name,
}
with open(RUN_DIR / 'run_info.json', 'w') as f:
    json.dump(run_info, f, indent=2, default=str)

print(f'Salvo em:')
print(f'  modelo + PCA + scaler : {MODEL_DIR}/')
print(f'  predicoes + bins + study: {RUN_DIR}/')
print(f'  optuna DB (retomavel)   : {storage}')